In [2]:
import pandas as pd
import os
import sys
import json
from pathlib import Path
import numpy as np

# Notebook is in notebooks/, so repo root is parent
REPO_ROOT = Path.cwd().parent
SRC_PATH = REPO_ROOT / "src"

# Insert src at the front of sys.path so imports work
sys.path.insert(0, str(SRC_PATH))

In [3]:
import importlib
import preprocessing.ecg_preprocessing as ep

importlib.reload(ep)

<module 'preprocessing.ecg_preprocessing' from '/Users/brandonng/Documents/GitHub/ClinicalDigitalTwin/src/preprocessing/ecg_preprocessing.py'>

In [4]:
from preprocessing.ecg_preprocessing import load_ecg_data

# Get repo root relative to the current notebook
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Load static preprocessing config
config_path = os.path.join(repo_root, "configs", "ecg_preprocessing_params.json")
with open(config_path, "r") as f:
    config = json.load(f)

# Set input and output directories
raw_dir = os.path.join(repo_root, config["paths"]["in_dir"])
processed_dir = os.path.join(repo_root, config["paths"]["processed_dir"])

# Pass directories to load function
ecg_record_list_df, machine_measurements_df, encounter_df = load_ecg_data(raw_dir, processed_dir, config)

In [5]:
machine_measurements_df.head(5)

,subject_id,study_id,cart_id,ecg_time,report_0,report_1,report_2,report_3,report_4,report_5,...,filtering,rr_interval,p_onset,p_end,qrs_onset,qrs_end,t_end,p_axis,qrs_axis,t_axis
0,10000032,40689238,6848296,2180-07-23 08:44:00,Sinus rhythm,Possible right atrial abnormality,NaN,Borderline ECG,NaN,NaN,...,60 Hz notch Baseline filter,659,40,128,170,258,518,81,77,79
1,10000032,44458630,6848296,2180-07-23 09:54:00,Sinus rhythm,Possible right atrial abnormality,NaN,Borderline ECG,NaN,NaN,...,60 Hz notch Baseline filter,722,40,124,162,246,504,77,75,70
2,10000032,49036311,6376932,2180-08-06 09:07:00,Sinus tachycardia,NaN,Normal ECG except for rate,NaN,NaN,NaN,...,60 Hz notch Baseline filter,600,40,130,162,244,474,79,72,77
3,10000117,45090959,6214760,2181-03-04 17:14:00,Sinus rhythm,NaN,Normal ECG,NaN,NaN,NaN,...,60 Hz notch Baseline filter,659,40,146,180,254,538,79,66,69
4,10000117,48446569,6632385,2183-09-18 13:52:00,Sinus rhythm,NaN,NaN,NaN,NaN,NaN,...,<not specified>,659,368,29999,504,590,868,84,80,77


# Machine Measurements

In [6]:
from preprocessing.ecg_preprocessing import preprocess_ecg_reports

cleaned_machine_measurements = preprocess_ecg_reports(machine_measurements_df)
cleaned_machine_measurements.head(5)

,subject_id,study_id,cart_id,ecg_time,bandwidth,filtering,rr_interval,p_onset,p_end,qrs_onset,qrs_end,t_end,p_axis,qrs_axis,t_axis,full_report
0,10000032,40689238,6848296,2180-07-23 08:44:00,0.005-150 Hz,60 Hz notch Baseline filter,659,40,128,170,258,518,81,77,79,"[sinus rhythm, possible right atrial abnormali..."
1,10000032,44458630,6848296,2180-07-23 09:54:00,0.005-150 Hz,60 Hz notch Baseline filter,722,40,124,162,246,504,77,75,70,"[sinus rhythm, possible right atrial abnormali..."
2,10000032,49036311,6376932,2180-08-06 09:07:00,0.005-150 Hz,60 Hz notch Baseline filter,600,40,130,162,244,474,79,72,77,"[sinus tachycardia, na, normal ecg except for ..."
3,10000117,45090959,6214760,2181-03-04 17:14:00,0.005-150 Hz,60 Hz notch Baseline filter,659,40,146,180,254,538,79,66,69,"[sinus rhythm, na, normal ecg, na, na, na, na,..."
4,10000117,48446569,6632385,2183-09-18 13:52:00,0.0005-150 Hz,<not specified>,659,368,29999,504,590,868,84,80,77,"[sinus rhythm, na, na, na, na, na, na, na, na,..."


In [7]:
from preprocessing.ecg_preprocessing import apply_phrase_labels

final_machine_measurements = apply_phrase_labels(cleaned_machine_measurements)
final_machine_measurements.head(5)

,subject_id,study_id,cart_id,ecg_time,bandwidth,filtering,rr_interval,p_onset,p_end,qrs_onset,...,junctional_rhythm,undetermined_rhythm,wpw_pattern,technical_error,lead_reversal,pericarditis,hyperkalemia,digitalis_effect,abnormal_p_waves,ventricular_response
0,10000032,40689238,6848296,2180-07-23 08:44:00,0.005-150 Hz,60 Hz notch Baseline filter,659,40,128,170,...,0,0,0,0,0,0,0,0,0,0
1,10000032,44458630,6848296,2180-07-23 09:54:00,0.005-150 Hz,60 Hz notch Baseline filter,722,40,124,162,...,0,0,0,0,0,0,0,0,0,0
2,10000032,49036311,6376932,2180-08-06 09:07:00,0.005-150 Hz,60 Hz notch Baseline filter,600,40,130,162,...,0,0,0,0,0,0,0,0,0,0
3,10000117,45090959,6214760,2181-03-04 17:14:00,0.005-150 Hz,60 Hz notch Baseline filter,659,40,146,180,...,0,0,0,0,0,0,0,0,0,0
4,10000117,48446569,6632385,2183-09-18 13:52:00,0.0005-150 Hz,<not specified>,659,368,29999,504,...,0,0,0,0,0,0,0,0,0,0


In [8]:
final_machine_measurements.columns[16:]

Index(['sinus_rhythm', 'sinus_bradycardia', 'sinus_tachycardia',
       'atrial_fibrillation', 'atrial_flutter', 'supraventricular_tachycardia',
       'ventricular_tachycardia', 'idioventricular_rhythm',
       'bundle_branch_block', 'right_bundle_branch_block',
       'left_bundle_branch_block', 'av_block',
       'left_anterior_fascicular_block', 'left_posterior_fascicular_block',
       'intraventricular_conduction_delay', 'aberrant_conduction',
       'left_ventricular_hypertrophy', 'right_ventricular_hypertrophy',
       'biatrial_enlargement', 'left_atrial_enlargement',
       'right_atrial_enlargement', 'axis_deviation', 'low_qrs_voltage',
       'high_qrs_voltage', 'st_t_abnormality', 'poor_r_wave_progression',
       'st_elevation', 'st_depression', 't_wave_inversion', 'prolonged_qt',
       'qt_prolongation', 'myocardial_ischemia', 'myocardial_injury',
       'infarct_pattern', 'pvcs', 'pacs', 'ventricular_bigeminy',
       'ventricular_trigeminy', 'supraventricular_bigeminy

In [9]:
def count_total_elements(full_report):
    """Count total elements using list comprehension."""
    return sum(len(arr) for arr in full_report)

count_total_elements(final_machine_measurements['full_report'])

14400144

In [10]:
def extract_unique_elements_to_file(full_report, output_file='unique_elements.txt', sort=True):
    """Extract unique elements using pandas explode method."""
    import pandas as pd
    
    # Explode the series to flatten all arrays
    unique_elements = full_report.explode().dropna().unique()
    
    # Sort if requested
    if sort:
        unique_elements = sorted(unique_elements)
    
    # Write to file
    with open(output_file, 'w', encoding='utf-8') as f:
        for element in unique_elements:
            f.write(f"{element}\n")
    
    print(f"Wrote {len(unique_elements)} unique elements to {output_file}")
    return set(unique_elements)

# all_elements = extract_unique_elements_to_file(final_machine_measurements['full_report'])


In [11]:
col_sums = final_machine_measurements[final_machine_measurements.columns[16:]].sum()
print(col_sums[col_sums < 1500].index.tolist())
print(col_sums[(col_sums < 1500) & (col_sums > 1)].sort_values(ascending=True))

['supraventricular_tachycardia', 'ventricular_tachycardia', 'idioventricular_rhythm', 'bundle_branch_block', 'av_block', 'left_posterior_fascicular_block', 'intraventricular_conduction_delay', 'aberrant_conduction', 'biatrial_enlargement', 'right_atrial_enlargement', 'low_qrs_voltage', 'high_qrs_voltage', 'st_t_abnormality', 'poor_r_wave_progression', 'st_elevation', 'st_depression', 't_wave_inversion', 'qt_prolongation', 'myocardial_ischemia', 'myocardial_injury', 'infarct_pattern', 'pvcs', 'ventricular_bigeminy', 'ventricular_trigeminy', 'supraventricular_bigeminy', 'ventricular_couplets', 'fusion_complexes', 'atrial_paced_rhythm', 'dual_paced_rhythm', 'ectopic_atrial_rhythm', 'junctional_rhythm', 'wpw_pattern', 'technical_error', 'lead_reversal', 'pericarditis', 'hyperkalemia', 'digitalis_effect', 'abnormal_p_waves', 'ventricular_response']
infarct_pattern                         3
st_elevation                            4
pericarditis                            5
intraventricular_c

In [12]:
print(14400144 - 1397569)

13002575


# Record List

In [13]:
ecg_record_list_df[ecg_record_list_df['subject_id']=='10000032'].head(5)

,subject_id,study_id,file_name,ecg_time,path
0,10000032,40689238,40689238,2180-07-23 08:44:00,files/p1000/p10000032/s40689238/40689238
1,10000032,44458630,44458630,2180-07-23 09:54:00,files/p1000/p10000032/s44458630/44458630
2,10000032,49036311,49036311,2180-08-06 09:07:00,files/p1000/p10000032/s49036311/49036311


In [14]:
encounter_df[encounter_df['subject_id'] == '10000032'].head(10)

,subject_id,hadm_id,ed_stay_id,ed_intime,ed_outtime,hosp_admittime,hosp_dischtime,race,gender,anchor_age,death_time,icu_stay_id,icu_intime,icu_outtime,icu_count,diagnosis,icd_codes
11981,10000032,29079034.0,39399961.0,2180-07-23 05:54:00,2180-07-23 14:00:00,2180-07-23 12:35:00,2180-07-25 17:55:00,WHITE,F,52.0,2180-09-09,[39553978],['2180-07-23 14:00:00'],['2180-07-23 23:50:47'],1.0,"['Other iatrogenic hypotension', 'Chronic hepa...","['45829', '07044', '7994', '2761', '78959', '2..."
164445,10000032,22595853.0,33258284.0,2180-05-06 19:17:00,2180-05-06 23:30:00,2180-05-06 22:23:00,2180-05-07 17:15:00,WHITE,F,52.0,2180-09-09 00:00:00,NaN,NaN,NaN,NaN,"['Portal hypertension', 'Other ascites', 'Cirr...","['5723', '78959', '5715', '07070', '496', '296..."
164446,10000032,22841357.0,38112554.0,2180-06-26 15:54:00,2180-06-26 21:31:00,2180-06-26 18:27:00,2180-06-27 18:49:00,WHITE,F,52.0,2180-09-09 00:00:00,NaN,NaN,NaN,NaN,['Unspecified viral hepatitis C with hepatic c...,"['07071', '78959', '2875', '2761', '496', '571..."
164447,10000032,25742920.0,35968195.0,2180-08-05 20:58:00,2180-08-06 01:44:00,2180-08-05 23:44:00,2180-08-07 17:50:00,WHITE,F,52.0,2180-09-09 00:00:00,NaN,NaN,NaN,NaN,['Chronic hepatitis C without mention of hepat...,"['07054', '78959', 'V462', '5715', '2767', '27..."


In [15]:
from preprocessing.ecg_preprocessing import match_ecg_to_encounters

merged = match_ecg_to_encounters(ecg_record_list_df, encounter_df)
merged.head(5)


,subject_id,study_id,file_name,ecg_time,path,hadm_id,ed_stay_id,icu_stay_id,icu_intime,icu_outtime,in_hosp,in_ed
4146579,16598616,40000035,40000035,2126-04-01 19:32:00,files/p1659/p16598616/s40000035/40000035,26101986.0,32731561.0,NaN,NaN,NaN,0,1
5286799,18370366,40000084,40000084,2179-08-30 11:58:00,files/p1837/p18370366/s40000084/40000084,NaN,37736561.0,NaN,NaN,NaN,0,1
1655937,12576058,40000115,40000115,2163-04-17 16:45:00,files/p1257/p12576058/s40000115/40000115,NaN,36567247.0,NaN,NaN,NaN,0,1
3004465,14691089,40000143,40000143,2175-07-21 01:01:00,files/p1469/p14691089/s40000143/40000143,23760732.0,38981935.0,NaN,NaN,NaN,0,1
3850042,16089780,40000152,40000152,2137-01-29 19:08:00,files/p1608/p16089780/s40000152/40000152,NaN,31326149.0,NaN,NaN,NaN,0,1


In [16]:
merged[merged['subject_id'] == '10000032']

,subject_id,study_id,file_name,ecg_time,path,hadm_id,ed_stay_id,icu_stay_id,icu_intime,icu_outtime,in_hosp,in_ed
0,10000032,40689238,40689238,2180-07-23 08:44:00,files/p1000/p10000032/s40689238/40689238,29079034.0,39399961.0,[39553978],['2180-07-23 14:00:00'],['2180-07-23 23:50:47'],0,1
4,10000032,44458630,44458630,2180-07-23 09:54:00,files/p1000/p10000032/s44458630/44458630,29079034.0,39399961.0,[39553978],['2180-07-23 14:00:00'],['2180-07-23 23:50:47'],0,1
11,10000032,49036311,49036311,2180-08-06 09:07:00,files/p1000/p10000032/s49036311/49036311,25742920.0,35968195.0,NaN,NaN,NaN,1,0


In [17]:
from preprocessing.ecg_preprocessing import add_icu_indicator

final_ecg_record_list = add_icu_indicator(merged)
final_ecg_record_list.head(5)

,subject_id,study_id,file_name,ecg_time,path,hadm_id,ed_stay_id,icu_stay_id,in_hosp,in_ed,in_icu,icu_within_12hrs,icu_within_24hrs
4146579,16598616,40000035,40000035,2126-04-01 19:32:00,files/p1659/p16598616/s40000035/40000035,26101986.0,32731561.0,NaN,0,1,0,0,0
5286799,18370366,40000084,40000084,2179-08-30 11:58:00,files/p1837/p18370366/s40000084/40000084,NaN,37736561.0,NaN,0,1,0,0,0
1655937,12576058,40000115,40000115,2163-04-17 16:45:00,files/p1257/p12576058/s40000115/40000115,NaN,36567247.0,NaN,0,1,0,0,0
3004465,14691089,40000143,40000143,2175-07-21 01:01:00,files/p1469/p14691089/s40000143/40000143,23760732.0,38981935.0,NaN,0,1,0,0,0
3850042,16089780,40000152,40000152,2137-01-29 19:08:00,files/p1608/p16089780/s40000152/40000152,NaN,31326149.0,NaN,0,1,0,0,0


In [18]:
final_ecg_record_list[final_ecg_record_list['subject_id'] == '10000032']

,subject_id,study_id,file_name,ecg_time,path,hadm_id,ed_stay_id,icu_stay_id,in_hosp,in_ed,in_icu,icu_within_12hrs,icu_within_24hrs
0,10000032,40689238,40689238,2180-07-23 08:44:00,files/p1000/p10000032/s40689238/40689238,29079034.0,39399961.0,[39553978],0,1,0,1,1
4,10000032,44458630,44458630,2180-07-23 09:54:00,files/p1000/p10000032/s44458630/44458630,29079034.0,39399961.0,[39553978],0,1,0,1,1
11,10000032,49036311,49036311,2180-08-06 09:07:00,files/p1000/p10000032/s49036311/49036311,25742920.0,35968195.0,NaN,1,0,0,0,0


In [19]:
final_ecg_record_list[['icu_within_12hrs', 'icu_within_24hrs']].sum()


icu_within_12hrs    23400
icu_within_24hrs    27644
dtype: int64

In [20]:
# Test case 1: ECG during ICU stay (should be in_icu=1)
test_df1 = pd.DataFrame({
    'ecg_time': [pd.Timestamp('2180-07-23 18:00:00')],
    'in_hosp': [1],
    'icu_intime': [['2180-07-23 14:00:00']],
    'icu_outtime': [['2180-07-23 23:50:47']]
})

# Test case 2: ECG outside ICU stay (should be in_icu=0)
test_df2 = pd.DataFrame({
    'ecg_time': [pd.Timestamp('2180-07-24 10:00:00')],
    'in_hosp': [1],
    'icu_intime': [['2180-07-23 14:00:00']],
    'icu_outtime': [['2180-07-23 23:50:47']]
})

# Test case 3: Not in hospital (should be in_icu=0)
test_df3 = pd.DataFrame({
    'ecg_time': [pd.Timestamp('2180-07-23 18:00:00')],
    'in_hosp': [0],
    'icu_intime': [['2180-07-23 14:00:00']],
    'icu_outtime': [['2180-07-23 23:50:47']]
})

# Test case 4: Multiple ICU stays, ECG in second stay (should be in_icu=1)
test_df4 = pd.DataFrame({
    'ecg_time': [pd.Timestamp('2180-07-25 15:00:00')],
    'in_hosp': [1],
    'icu_intime': [['2180-07-23 14:00:00', '2180-07-25 10:00:00']],
    'icu_outtime': [['2180-07-23 23:50:47', '2180-07-25 20:00:00']]
})

# Test case 5: No ICU data (should be in_icu=0)
test_df5 = pd.DataFrame({
    'ecg_time': [pd.Timestamp('2180-07-23 18:00:00')],
    'in_hosp': [1],
    'icu_intime': [np.nan],
    'icu_outtime': [np.nan]
})

# Test case 6: ICU admission 6 hours after ECG (should be icu_within_12hrs=1, icu_within_24hrs=1)
test_df6 = pd.DataFrame({
    'ecg_time': [pd.Timestamp('2180-07-23 08:00:00')],
    'in_hosp': [1],
    'icu_intime': [['2180-07-23 14:00:00']],  # 6 hours later
    'icu_outtime': [['2180-07-23 23:50:47']]
})

# Test case 7: ICU admission 18 hours after ECG (should be icu_within_12hrs=0, icu_within_24hrs=1)
test_df7 = pd.DataFrame({
    'ecg_time': [pd.Timestamp('2180-07-23 08:00:00')],
    'in_hosp': [1],
    'icu_intime': [['2180-07-24 02:00:00']],  # 18 hours later
    'icu_outtime': [['2180-07-24 12:00:00']]
})

# Test case 8: ICU admission 30 hours after ECG (should be icu_within_12hrs=0, icu_within_24hrs=0)
test_df8 = pd.DataFrame({
    'ecg_time': [pd.Timestamp('2180-07-23 08:00:00')],
    'in_hosp': [1],
    'icu_intime': [['2180-07-24 14:00:00']],  # 30 hours later
    'icu_outtime': [['2180-07-24 20:00:00']]
})

# Test case 9: ICU before ECG (should be all 0)
test_df9 = pd.DataFrame({
    'ecg_time': [pd.Timestamp('2180-07-23 20:00:00')],
    'in_hosp': [1],
    'icu_intime': [['2180-07-23 14:00:00']],
    'icu_outtime': [['2180-07-23 18:00:00']]  # Ended before ECG
})

# Test case 10: ED case (in_hosp=0) with ICU 6 hours later (should be icu_within_12hrs=1, icu_within_24hrs=1)
test_df10 = pd.DataFrame({
    'ecg_time': [pd.Timestamp('2180-07-23 08:00:00')],
    'in_hosp': [0],  # ED case
    'icu_intime': [['2180-07-23 14:00:00']],  # 6 hours later
    'icu_outtime': [['2180-07-23 23:50:47']]
})

# Run tests
print("Test 1 (ECG during ICU):")
result1 = add_icu_indicator(test_df1)
print(result1[['in_icu', 'icu_within_12hrs', 'icu_within_24hrs']])
print(f"Expected: in_icu=1, icu_within_12hrs=0, icu_within_24hrs=0\n")

print("Test 2 (ECG outside ICU):")
result2 = add_icu_indicator(test_df2)
print(result2[['in_icu', 'icu_within_12hrs', 'icu_within_24hrs']])
print(f"Expected: in_icu=0, icu_within_12hrs=0, icu_within_24hrs=0\n")

print("Test 3 (Not in hospital):")
result3 = add_icu_indicator(test_df3)
print(result3[['in_icu', 'icu_within_12hrs', 'icu_within_24hrs']])
print(f"Expected: in_icu=0, icu_within_12hrs=0, icu_within_24hrs=0\n")

print("Test 4 (Multiple ICU stays, in second):")
result4 = add_icu_indicator(test_df4)
print(result4[['in_icu', 'icu_within_12hrs', 'icu_within_24hrs']])
print(f"Expected: in_icu=1, icu_within_12hrs=0, icu_within_24hrs=0\n")

print("Test 5 (No ICU data):")
result5 = add_icu_indicator(test_df5)
print(result5[['in_icu', 'icu_within_12hrs', 'icu_within_24hrs']])
print(f"Expected: in_icu=0, icu_within_12hrs=0, icu_within_24hrs=0\n")

Test 1 (ECG during ICU):
   in_icu  icu_within_12hrs  icu_within_24hrs
0       1                 0                 0
Expected: in_icu=1, icu_within_12hrs=0, icu_within_24hrs=0

Test 2 (ECG outside ICU):
   in_icu  icu_within_12hrs  icu_within_24hrs
0       0                 0                 0
Expected: in_icu=0, icu_within_12hrs=0, icu_within_24hrs=0

Test 3 (Not in hospital):
   in_icu  icu_within_12hrs  icu_within_24hrs
0       0                 0                 0
Expected: in_icu=0, icu_within_12hrs=0, icu_within_24hrs=0

Test 4 (Multiple ICU stays, in second):
   in_icu  icu_within_12hrs  icu_within_24hrs
0       1                 0                 0
Expected: in_icu=1, icu_within_12hrs=0, icu_within_24hrs=0

Test 5 (No ICU data):
   in_icu  icu_within_12hrs  icu_within_24hrs
0       0                 0                 0
Expected: in_icu=0, icu_within_12hrs=0, icu_within_24hrs=0



In [21]:
print("Test 6 (ICU 6 hrs after ECG):")
result6 = add_icu_indicator(test_df6)
print(result6[['in_icu', 'icu_within_12hrs', 'icu_within_24hrs']])
print(f"Expected: in_icu=0, icu_within_12hrs=1, icu_within_24hrs=1\n")

print("Test 7 (ICU 18 hrs after ECG):")
result7 = add_icu_indicator(test_df7)
print(result7[['in_icu', 'icu_within_12hrs', 'icu_within_24hrs']])
print(f"Expected: in_icu=0, icu_within_12hrs=0, icu_within_24hrs=1\n")

print("Test 8 (ICU 30 hrs after ECG):")
result8 = add_icu_indicator(test_df8)
print(result8[['in_icu', 'icu_within_12hrs', 'icu_within_24hrs']])
print(f"Expected: in_icu=0, icu_within_12hrs=0, icu_within_24hrs=0\n")

print("Test 9 (ICU before ECG):")
result9 = add_icu_indicator(test_df9)
print(result9[['in_icu', 'icu_within_12hrs', 'icu_within_24hrs']])
print(f"Expected: in_icu=0, icu_within_12hrs=0, icu_within_24hrs=0\n")

print("Test 10 (ED case with ICU 6 hrs later):")
result10 = add_icu_indicator(test_df10)
print(result10[['in_icu', 'icu_within_12hrs', 'icu_within_24hrs']])
print(f"Expected: in_icu=0, icu_within_12hrs=1, icu_within_24hrs=1\n")

Test 6 (ICU 6 hrs after ECG):
   in_icu  icu_within_12hrs  icu_within_24hrs
0       0                 1                 1
Expected: in_icu=0, icu_within_12hrs=1, icu_within_24hrs=1

Test 7 (ICU 18 hrs after ECG):
   in_icu  icu_within_12hrs  icu_within_24hrs
0       0                 0                 1
Expected: in_icu=0, icu_within_12hrs=0, icu_within_24hrs=1

Test 8 (ICU 30 hrs after ECG):
   in_icu  icu_within_12hrs  icu_within_24hrs
0       0                 0                 0
Expected: in_icu=0, icu_within_12hrs=0, icu_within_24hrs=0

Test 9 (ICU before ECG):
   in_icu  icu_within_12hrs  icu_within_24hrs
0       0                 0                 0
Expected: in_icu=0, icu_within_12hrs=0, icu_within_24hrs=0

Test 10 (ED case with ICU 6 hrs later):
   in_icu  icu_within_12hrs  icu_within_24hrs
0       0                 1                 1
Expected: in_icu=0, icu_within_12hrs=1, icu_within_24hrs=1



# ECG Dataset

In [22]:
print(final_ecg_record_list.shape)
final_ecg_record_list.head(5)

(411120, 13)


,subject_id,study_id,file_name,ecg_time,path,hadm_id,ed_stay_id,icu_stay_id,in_hosp,in_ed,in_icu,icu_within_12hrs,icu_within_24hrs
4146579,16598616,40000035,40000035,2126-04-01 19:32:00,files/p1659/p16598616/s40000035/40000035,26101986.0,32731561.0,NaN,0,1,0,0,0
5286799,18370366,40000084,40000084,2179-08-30 11:58:00,files/p1837/p18370366/s40000084/40000084,NaN,37736561.0,NaN,0,1,0,0,0
1655937,12576058,40000115,40000115,2163-04-17 16:45:00,files/p1257/p12576058/s40000115/40000115,NaN,36567247.0,NaN,0,1,0,0,0
3004465,14691089,40000143,40000143,2175-07-21 01:01:00,files/p1469/p14691089/s40000143/40000143,23760732.0,38981935.0,NaN,0,1,0,0,0
3850042,16089780,40000152,40000152,2137-01-29 19:08:00,files/p1608/p16089780/s40000152/40000152,NaN,31326149.0,NaN,0,1,0,0,0


In [23]:
print(final_machine_measurements.shape)
final_machine_measurements.head(5)

(800008, 75)


,subject_id,study_id,cart_id,ecg_time,bandwidth,filtering,rr_interval,p_onset,p_end,qrs_onset,...,junctional_rhythm,undetermined_rhythm,wpw_pattern,technical_error,lead_reversal,pericarditis,hyperkalemia,digitalis_effect,abnormal_p_waves,ventricular_response
0,10000032,40689238,6848296,2180-07-23 08:44:00,0.005-150 Hz,60 Hz notch Baseline filter,659,40,128,170,...,0,0,0,0,0,0,0,0,0,0
1,10000032,44458630,6848296,2180-07-23 09:54:00,0.005-150 Hz,60 Hz notch Baseline filter,722,40,124,162,...,0,0,0,0,0,0,0,0,0,0
2,10000032,49036311,6376932,2180-08-06 09:07:00,0.005-150 Hz,60 Hz notch Baseline filter,600,40,130,162,...,0,0,0,0,0,0,0,0,0,0
3,10000117,45090959,6214760,2181-03-04 17:14:00,0.005-150 Hz,60 Hz notch Baseline filter,659,40,146,180,...,0,0,0,0,0,0,0,0,0,0
4,10000117,48446569,6632385,2183-09-18 13:52:00,0.0005-150 Hz,<not specified>,659,368,29999,504,...,0,0,0,0,0,0,0,0,0,0


In [24]:
ecg = final_ecg_record_list.merge(final_machine_measurements, on=['subject_id', 'study_id', 'ecg_time'], how='inner')

print(ecg.shape)
ecg.head(5)

(403121, 85)


,subject_id,study_id,file_name,ecg_time,path,hadm_id,ed_stay_id,icu_stay_id,in_hosp,in_ed,...,junctional_rhythm,undetermined_rhythm,wpw_pattern,technical_error,lead_reversal,pericarditis,hyperkalemia,digitalis_effect,abnormal_p_waves,ventricular_response
0,16598616,40000035,40000035,2126-04-01 19:32:00,files/p1659/p16598616/s40000035/40000035,26101986.0,32731561.0,NaN,0,1,...,0,0,0,0,0,0,0,0,0,0
1,18370366,40000084,40000084,2179-08-30 11:58:00,files/p1837/p18370366/s40000084/40000084,NaN,37736561.0,NaN,0,1,...,0,0,0,0,0,0,0,0,0,0
2,12576058,40000115,40000115,2163-04-17 16:45:00,files/p1257/p12576058/s40000115/40000115,NaN,36567247.0,NaN,0,1,...,0,0,0,0,0,0,0,0,0,0
3,14691089,40000143,40000143,2175-07-21 01:01:00,files/p1469/p14691089/s40000143/40000143,23760732.0,38981935.0,NaN,0,1,...,0,0,0,0,0,0,0,0,0,0
4,16089780,40000152,40000152,2137-01-29 19:08:00,files/p1608/p16089780/s40000152/40000152,NaN,31326149.0,NaN,0,1,...,0,0,0,0,0,0,0,0,0,0


In [25]:
print(f'Total Hours of ECG recordings: {ecg.shape[0] * 10 / 60 / 60}')

Total Hours of ECG recordings: 1119.7805555555556


In [26]:
print(f'Total Number of ECG recordings in ED: {sum(ecg['in_ed'])}')
print(f'Total Hours of ECG recordings in ED: {sum(ecg['in_ed']) * 10 / 60 / 60}')

Total Number of ECG recordings in ED: 158284
Total Hours of ECG recordings in ED: 439.6777777777778


In [27]:
print(f'Total Number of ECG recordings in ED: {sum(ecg['in_hosp'])}')
print(f'Total Hours of ECG recordings in ED: {sum(ecg['in_hosp']) * 10 / 60 / 60}')

Total Number of ECG recordings in ED: 259465
Total Hours of ECG recordings in ED: 720.7361111111111


In [29]:
ecg.columns

Index(['subject_id', 'study_id', 'file_name', 'ecg_time', 'path', 'hadm_id',
       'ed_stay_id', 'icu_stay_id', 'in_hosp', 'in_ed', 'in_icu',
       'icu_within_12hrs', 'icu_within_24hrs', 'cart_id', 'bandwidth',
       'filtering', 'rr_interval', 'p_onset', 'p_end', 'qrs_onset', 'qrs_end',
       't_end', 'p_axis', 'qrs_axis', 't_axis', 'full_report', 'sinus_rhythm',
       'sinus_bradycardia', 'sinus_tachycardia', 'atrial_fibrillation',
       'atrial_flutter', 'supraventricular_tachycardia',
       'ventricular_tachycardia', 'idioventricular_rhythm',
       'bundle_branch_block', 'right_bundle_branch_block',
       'left_bundle_branch_block', 'av_block',
       'left_anterior_fascicular_block', 'left_posterior_fascicular_block',
       'intraventricular_conduction_delay', 'aberrant_conduction',
       'left_ventricular_hypertrophy', 'right_ventricular_hypertrophy',
       'biatrial_enlargement', 'left_atrial_enlargement',
       'right_atrial_enlargement', 'axis_deviation', 'low_qr

In [33]:
ecg.loc[ecg['technical_error'] == 1, ['path', 'rr_interval', 'p_onset', 'p_end', 'qrs_onset', 'qrs_end',
                                      't_end', 'p_axis', 'qrs_axis', 't_axis']]


,path,rr_interval,p_onset,p_end,qrs_onset,qrs_end,t_end,p_axis,qrs_axis,t_axis
1674,files/p1075/p10757372/s40043165/40043165,0,29999,29999,0,29999,29999,0,0,32767
1862,files/p1513/p15137987/s40047971/40047971,29999,29999,29999,29999,29999,29999,29999,29999,29999
3661,files/p1334/p13347162/s40093146/40093146,29999,29999,29999,29999,29999,29999,29999,29999,29999
4851,files/p1256/p12567301/s40122085/40122085,29999,29999,29999,29999,29999,29999,29999,29999,29999
6320,files/p1009/p10093718/s40158485/40158485,0,29999,29999,0,29999,29999,0,0,32767
...,...,...,...,...,...,...,...,...,...,...
397300,files/p1606/p16066745/s49852093/49852093,0,29999,29999,0,29999,29999,0,0,32767
397404,files/p1531/p15318682/s49854690/49854690,0,29999,29999,0,29999,29999,0,0,32767
398141,files/p1188/p11880923/s49873908/49873908,29999,29999,29999,29999,29999,29999,29999,29999,29999
400954,files/p1581/p15816613/s49944705/49944705,29999,694,32608,2412,716,3310,32496,8742,0
